# Assignment:4
1. Create a supervisor node.
2. Create one router function
3. Create three more node
    - 3.1 llm call (llm node)
    - 3.2 RAG (rag node)
    - 3.3 web crawler(fetch the info in realtime from internet)
4. Created one more node after this for validation of generated output --> explore the validation part how to do that
5. if validation going to be failed in that case again go to supervioser node and then supervisor node will again decide what needs to be call next
6. once the validation will pass then only generate the final output

In [16]:
from dotenv import load_dotenv
from langchain_groq.chat_models import ChatGroq
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
# from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage

from langchain_core.globals import set_debug

set_debug(True)        # Most detailed logs

In [2]:
load_dotenv()

True

In [3]:
# instantialing llm
llm = ChatGroq(model="qwen/qwen3-32b", reasoning_effort="none")

In [4]:
# result = llm.invoke("What is the capital of France?")

In [5]:
# print(result)

### Memory

In [ ]:
# Defining state

class State(TypedDict):
    messages: 
    

In [9]:
state["messages"] = []

In [11]:
state['messages'].append("heello, whats up?")

In [13]:
state['messages'].append("what is the capital of France?")

In [14]:
state

{'messages': ['heello, whats up?', 'what is the capital of France?']}

### Supervisor 

In [ ]:
# creating a supervisor node
def supervisor_node(input):
    '''This is a supervisor node that reviews the input and take the necessary action or redirect the call to the appropriate node.'''
    # defining prompt
    supervisor_prompt = ChatPromptTemplate.from_messages(
        [
            ("system","You are a supervisor node in a langgraph based system. Please review the input by user and respond accordingly."),
            ("human", "{input}")
        ]
    )

    # invoking llm with the prompt 
    supervisor_chain = supervisor_prompt | llm
    supervisor_response = supervisor_chain.invoke({"input": input})
    return supervisor_response

In [21]:
result = supervisor_node("what is the capital of France?")

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "what is the capital of France?"
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "input": "what is the capital of France?"
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatGroq] Entering LLM run with input:
{
  "prompts": [
    "System: You are a supervisor node in a langgraph based system. Please review the input by user and provide guidance.\nHuman: what is the capital of France?"
  ]
}
[llm/end] [chain:RunnableSequence > llm:ChatGroq] s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "The capital of France is Paris.",
        "generation_info": {
          "finish_reason": "stop",
          "logprobs": null
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type

In [8]:
print(result)

content='The capital of France is Paris.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 46, 'total_tokens': 54, 'completion_time': 0.013966841, 'completion_tokens_details': None, 'prompt_time': 0.001625857, 'prompt_tokens_details': None, 'queue_time': 0.043510992, 'total_time': 0.015592698}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'reasoning_effort': 'none', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d6bbc-733f-7f33-9e8a-7069ca413751-0' usage_metadata={'input_tokens': 46, 'output_tokens': 8, 'total_tokens': 54}


In [9]:
# # Preparing for RAG node

# data = [r"C:\Users\deepak.a.dhiman\projects\agentic_ai\AgenticAI_2.0\data\Attention_Paper.pdf",
#         r"C:\Users\deepak.a.dhiman\projects\agentic_ai\AgenticAI_2.0\data\BERT_Paper.pdf"]

# chunks = []

# for document in data:
#     loader = PyPDFLoader(document)
#     documents = loader.load()
#     text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
#     chunks.extend(text_splitter.split_documents(documents))

# print(f"Total chunks created: {len(chunks)}")


### RAG Tool

In [24]:
# intialisng embdding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# # indexing the chunks using chroma vector store
# vector_store = Chroma.from_documents(chunks, 
#                                      embedding=embedding_model, 
#                                      persist_directory="./chroma_db",
#                                      )

vector_store = Chroma(persist_directory="./chroma_db", embedding_function=embedding_model)

retriever = vector_store.as_retriever(search_kwargs={"k": 5})

def rag_node(query):
    '''This is a RAG node that retrieves relevant information from the indexed chunks based on the query and provides an answer'''
    relevant_chunks = retriever.invoke(query)
    # defining prompt
    rag_prompt = ChatPromptTemplate.from_messages(
        [
            ("system","Answer the user query only from the context provided in the relevant chunks. If the relevant chunks do not contain the answer, say 'I don't know.'"),
            ("human", "Query: {query}\nRelevant Chunks: {relevant_chunks}")
        ]
    )

    # invoking llm with the prompt
    rag_chain = rag_prompt | llm
    rag_response = rag_chain.invoke({"query": query, "relevant_chunks": relevant_chunks })
    return rag_response
    

In [25]:
# result = rag_node("What is the attention mechanism in deep learning?")
# result = rag_node("What is BERT model in deep learning?")
result = rag_node("What is BERT stands for?")

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
[inputs]
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
[inputs]
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatGroq] Entering LLM run with input:
{
  "prompts": [
    "System: Answer the user query only from the context provided in the relevant chunks. If the relevant chunks do not contain the answer, say 'I don't know.'\nHuman: Query: What is BERT stands for?\nRelevant Chunks: [Document(id='7bb59180-40ed-4f58-9481-6e0dc1562e0b', metadata={'page_label': '4', 'title': '', 'moddate': '2019-05-28T00:07:51+00:00', 'creationdate': '2019-05-28T00:07:51+00:00', 'trapped': '/False', 'page': 3, 'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'total_pages': 16, 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) k

In [26]:
print(result.content)

I don't know.
